### PageIndex — Vectorless RAG

### Key Concept

### Traditional RAG → chunk → embed → cosine similarity → retrieve
### PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections

The problem with vector RAG:
Similarity ≠ Relevance
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.



### Section 1: Install & Setup

In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
PAGEINDEX_API_KEY = os.getenv('PAGEINDEX_API_KEY')

In [4]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

✅ PageIndex client ready
✅ OpenAI client ready


### Section 2: Upload & Index a PDF

What happens here:

- Upload your PDF to the PageIndex cloud
- PageIndex uses an LLM to read the document structure
- Builds a hierarchical tree index (like a smart Table of Contents)
- Returns a doc_id for all future operations

Why NO chunking?
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [8]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./employee-guide.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./employee-guide.pdf
✅ Uploaded!
📋 Document ID: pi-cmo2rvoex0l5r01r4d4d86ad9
   (Save this ID — you'll use it throughout the notebook)


In [9]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


### Section 3: Inspect the Tree Structure

Each node has:

- node_id — unique ID used during retrieval
- title — section heading
- page_index — page number in original PDF
- text — section summary (when node_summary=True)
- nodes — child sections (nested)
This structure is what the LLM reasons over during retrieval.

In [10]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 46

🌲 Raw tree (first node):
{
  "title": "Welcome to HMH!",
  "node_id": "0000",
  "page_index": 1,
  "summary": "This document introduces the HMH India Employee Guide, outlining the company's commitment to fostering a positive work environment and employee growth. It emphasizes the mandatory requirement for all employees to understand and comply with the guide's provisions, which are subject to annual review and potential revision by senior management. The guide is not exhaustive, and employees are also bound by their employment agreements. It serves as a tool for consistent behavior and fair policy administration, with provisions for addressing questions and communicating changes. The guide is company property and applies to all employees in India, covering essential employment-related guidelines.",
  "text": "# Welcome to HMH!\n\nThe goal of HMH Technology Private Limited (the \"Company\") is to create a great place to work and opportunities for professional a

In [11]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Welcome to HMH!  (p.1)
[0001] Contents  (p.2)
[0002] Code of Conduct  (p.3)
[0003] Employee Relations  (p.3)
[0004] Employee’s Duties and Responsibilities  (p.3)
[0005] Diversity and Inclusion  (p.3)
[0006] Equal Employment Opportunity  (p.3)
[0007] Prevention of Sexual Harassment  (p.4)
[0008] HIV and Aids  (p.4)
[0009] Employee Information and Records  (p.4)
[0010] Background Check Process  (p.4)
[0011] Employees on Probation  (p.5)
[0012] Personnel Records and Personal Information  (p.5)
[0013] Engagement with External Parties  (p.6)
[0014] Work Schedules, Attendance, and Absences  (p.6)
[0015] Flex-Time  (p.6)
[0016] Attendance  (p.7)
[0017] Remote Work Arrangements  (p.7)
[0018] Use of Company Property  (p.7)
[0019] Privacy, Search, and Surveillance  (p.8)
[0020] Acceptable Use Policy  (p.8)
[0021] Return of Company Property  (p.10)
[0022] Non-Employees  (p.11)
[0023] Employees  (p.11)
[0024] Exceptions  (p.11)
[0025] Safety and Hazards  (p.12)
[

In [12]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 46
   Each node = one retrievable section of the document


### Section 4: LLM Tree Search — The Core of PageIndex

This is where PageIndex fundamentally differs from vector RAG.

#### Vector RAG retrieval: query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks

#### PageIndex retrieval: query + tree → LLM reasons → "node 0007 and 0008 contain the answer"

The LLM acts like a human expert scanning a Table of Contents.

In [13]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [15]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the Resignation Process ?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the Resignation Process ?

🧠 LLM Reasoning:
To determine where the Resignation Process might be located in the document, first examine the document tree titles for any direct reference to resignation or closely related topics like employee exit, termination, or final procedures. While there is no direct match for a 'Resignation Process,' the title 'Exit Interviews' (node_id: 0043) suggests that it deals with procedures relating to employee termination or departure, including possibly the resignation process. This makes it the most relevant section. Furthermore, 'Return of Company Property' (node_id: 0021) could be related to final steps in the resignation process. Both sections are likely candidates for covering aspects of resigning from the company.

🎯 Selected Node IDs: ['0043', '0021']


### Section 5: Full End-to-End RAG Pipeline

3 steps:

- Tree Search → LLM picks relevant node_ids
- Retrieve → Fetch the actual section content from those nodes
- Generate → LLM writes a grounded answer with page citations

What makes this better than vector RAG:

- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly which section the answer comes from
- No hallucination from irrelevant chunks

In [16]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found


In [17]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [18]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [19]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is the policy for Prevention of Sexual Harassment?",
    tree=pageindex_tree
)

🔍 Query: What is the policy for Prevention of Sexual Harassment?

🧠 Reasoning: The query is asking about the policy for Prevention of Sexual Harassment. To address this, the document tree must be examined for a section that directly refers to this topic. Scanning through the tit...
🎯 Retrieved node IDs: ['0007']
📄 Sections found: ['Prevention of Sexual Harassment']

📝 Answer:
The policy for the Prevention of Sexual Harassment states that the company does not tolerate verbal or physical conduct that creates an intimidating, offensive, or hostile environment. Harassment, including sexual harassment, is prohibited, and every associate has the right to be protected against it. A detailed policy for the prevention of sexual harassment at the workplace is available from the company (Section: 'Prevention of Sexual Harassment' | Page 4).


### Section 6: Expert-Guided Retrieval

The killer feature no one talks about.

With vector RAG, injecting domain expertise requires fine-tuning the embedding model — expensive and time-consuming.

With PageIndex, you just add rules to the prompt:

- "If the query mentions EBITDA → prioritize the MD&A section"
- "If the query is about risks  → check Part I, Item 1A"
This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.

In [20]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [21]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# Krish Naik Academy | 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [22]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "gpt-4o"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [23]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Details of the modern llm finetuning?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: Details of the modern llm finetuning?

── Without Expert Rules ──
Nodes: []

── With Expert Rules ──
Nodes: []
Reasoning: The query asks for details on modern LLM finetuning. Based on the expert routing rules, inquiries about LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO, quantization, TRL, Unsloth, synthetic data, reasoning models, evaluation, and deployment related to finetuning should be routed to M9 (Modern LLM Fine


In [25]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="What is the policy for Prevention of Sexual Harassment? ",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

The policy for Prevention of Sexual Harassment states that the company does not tolerate verbal or physical conduct that creates an intimidating, offensive, or hostile environment and believes it undermines safety and dignity. Harassment of any kind, including sexual harassment, is forbidden, and every associate has the right to be protected against it (Prevention of Sexual Harassment, Page 4).


### Section 7: Chat API — Zero LLM Setup

When to use this:

- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

In [26]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here are the key highlights of the **HMH India Employee Guide**:

---

### 📋 Conduct & Ethics
- All employees must comply with the **HMH Global Code of Conduct** and complete related training.
- **Zero tolerance** for bribery, corruption, fraud, and cash/gift exchanges with external parties.
- Employees are expected to raise workplace concerns with their supervisor or HR in good faith.

### 🤝 Diversity & Equal Opportunity
- The company prohibits discrimination based on sex, race, caste, religion, gender identity, sexual orientation, disability, and more.
- Dedicated policies exist for **prevention of sexual harassment** and protection of persons affected by **HIV/AIDS**.

### 🗂️ Employment & Records
- All offers are contingent on **background checks** (identity, criminal, employment history, education).
- Employees must keep personal information (address, marital status, etc.) updated within specified timeframes.
- A **probation period** applies as per each employee'

In [27]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "What were the main revenue sources last year?",
    "How does that compare to the year before?",
    "What factors drove that change?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: What were the main revenue sources last year?
🤖 Assistant: The document you've shared — **employee-guide.pdf** — is the HMH India Employee Guide, which covers company policies, conduct, benefits, and work environment. It doesn't contain any financial or revenue information.

If you're looking for revenue data, you'd need a financial report or annual report. Would you like to upload one, or do you have a different question about the employee guide?...
-------------------------------------------------------

👤 User: How does that compare to the year before?
🤖 Assistant: As mentioned, the **employee-guide.pdf** doesn't contain any financial data, so there's no revenue information to compare — for either year.

If you'd like revenue comparisons, please share a financial or annual report and I'd be happy to help analyze it. Is there anything else I can help you with from the employee guide?...
-------------------------------------------------------

👤 User: What factors drove that 

### Section 8: Self-Hosted Open Source Option

Use this when:

- You don't want to send documents to any cloud
- You need full data privacy / on-prem deployment
- You want to inspect or customize the tree-building logic